# LSTM Training: Atlanta → Birmingham External Generalization

**Experiment design**

Atlanta is the only development city:
- 70% training
- 15% validation
- 15% internal test

Birmingham is held out as a completely external test city.

Birmingham must not be used for training, validation, hyperparameter tuning, early stopping, model selection, or scaler fitting.

**Model:** 64-unit LSTM → Dropout 0.30 → 32-unit LSTM → Dropout 0.30 → Dense 16 ReLU → Dense 1.

**Training:** Adam, MSE, batch size 64, maximum 100 epochs, EarlyStopping patience 10, ReduceLROnPlateau.


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================
import os
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)


In [ ]:
# ============================================================
# 2. GPU VERIFICATION
# ============================================================
gpus = tf.config.list_physical_devices("GPU")
print("GPU devices:", gpus)

if not gpus:
    raise RuntimeError("TensorFlow cannot see the GPU. Stop here and fix the GPU environment.")

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("GPU detected successfully.")


In [ ]:
# ============================================================
# 3. REPRODUCIBILITY
# ============================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
print("Random seed:", SEED)


In [ ]:
# ============================================================
# 4. CONFIGURATION
# ============================================================
DATA_DIR = ".."

ATLANTA_PREFIX = "atlanta"
BIRMINGHAM_PREFIX = "birmingham"

BATCH_SIZE = 64
EPOCHS = 100
SEQUENCE_LENGTH = 24

MODEL_PATH = os.path.join(DATA_DIR, "atlanta_lstm_best.keras")
ATLANTA_PREDICTIONS_PATH = os.path.join(DATA_DIR, "atlanta_test_predictions.csv")
BIRMINGHAM_PREDICTIONS_PATH = os.path.join(DATA_DIR, "birmingham_external_predictions.csv")

print("Data directory:", DATA_DIR)


In [ ]:
# ============================================================
# 5. LOAD PREPARED ARRAYS
# ============================================================
def load_city_arrays(prefix, data_dir):
    return (
        np.load(os.path.join(data_dir, f"{prefix}_X_train.npy")),
        np.load(os.path.join(data_dir, f"{prefix}_y_train.npy")),
        np.load(os.path.join(data_dir, f"{prefix}_X_val.npy")),
        np.load(os.path.join(data_dir, f"{prefix}_y_val.npy")),
        np.load(os.path.join(data_dir, f"{prefix}_X_test.npy")),
        np.load(os.path.join(data_dir, f"{prefix}_y_test.npy"))
    )

(
    X_train, y_train,
    X_val, y_val,
    X_atlanta_test, y_atlanta_test
) = load_city_arrays(ATLANTA_PREFIX, DATA_DIR)

(
    _X_birmingham_train, _y_birmingham_train,
    _X_birmingham_val, _y_birmingham_val,
    X_birmingham_test, y_birmingham_test
) = load_city_arrays(BIRMINGHAM_PREFIX, DATA_DIR)

print("Atlanta train:", X_train.shape, y_train.shape)
print("Atlanta validation:", X_val.shape, y_val.shape)
print("Atlanta test:", X_atlanta_test.shape, y_atlanta_test.shape)
print("Birmingham external test:", X_birmingham_test.shape, y_birmingham_test.shape)


## Scaling rule

The Atlanta scaler is the only scaler allowed in this experiment.

If the Birmingham `.npy` arrays were produced with Birmingham-fitted scalers, they must be recreated using the Atlanta scalers before using them here. This notebook assumes the Birmingham arrays have already been transformed using the Atlanta scalers.


In [ ]:
# ============================================================
# 6. LOAD ATLANTA SCALERS
# ============================================================
X_scaler = joblib.load(os.path.join(DATA_DIR, "atlanta_X_scaler.pkl"))
y_scaler = joblib.load(os.path.join(DATA_DIR, "atlanta_y_scaler.pkl"))

print("Atlanta scalers loaded.")


In [ ]:
# ============================================================
# 7. DATA SHAPE CHECKS
# ============================================================
n_features = X_train.shape[2]

assert X_train.shape[1] == SEQUENCE_LENGTH
assert X_val.shape[1] == SEQUENCE_LENGTH
assert X_atlanta_test.shape[1] == SEQUENCE_LENGTH
assert X_birmingham_test.shape[1] == SEQUENCE_LENGTH

assert X_val.shape[2] == n_features
assert X_atlanta_test.shape[2] == n_features
assert X_birmingham_test.shape[2] == n_features

print("Sequence length:", SEQUENCE_LENGTH)
print("Input features:", n_features)
print("Shape checks passed.")


In [ ]:
# ============================================================
# 8. BUILD MODEL
# ============================================================
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, n_features)),
    Dropout(0.30),
    LSTM(32),
    Dropout(0.30),
    Dense(16, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="mse",
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
)

model.summary()


In [ ]:
# ============================================================
# 9. CALLBACKS
# ============================================================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-7,
    verbose=1
)


In [ ]:
# ============================================================
# 10. TRAIN ON ATLANTA ONLY
# ============================================================
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

model.save(MODEL_PATH)
print("Best-weight model saved to:", MODEL_PATH)


# Overfitting diagnostics

The next plots are intended both for model diagnosis and for presentation.

A healthy training run should show training and validation loss decreasing together and then stabilizing. If training loss keeps decreasing while validation loss rises, that is evidence of overfitting.

The generalization gap below gives an additional numerical check.


In [ ]:
# ============================================================
# 11. TRAINING VS VALIDATION LOSS
# ============================================================
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_ran = range(1, len(loss) + 1)

best_epoch = int(np.argmin(val_loss)) + 1

plt.figure(figsize=(11, 5))
plt.plot(epochs_ran, loss, linewidth=2, label="Training loss")
plt.plot(epochs_ran, val_loss, linewidth=2, label="Validation loss")
plt.axvline(best_epoch, linestyle="--", linewidth=1.5, label=f"Best epoch = {best_epoch}")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("LSTM Training vs Validation Loss")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

print("Best validation epoch:", best_epoch)
print("Best validation loss:", min(val_loss))


In [ ]:
# ============================================================
# 12. TRAINING VS VALIDATION MAE
# ============================================================
mae = history.history["mae"]
val_mae = history.history["val_mae"]

plt.figure(figsize=(11, 5))
plt.plot(epochs_ran, mae, linewidth=2, label="Training MAE")
plt.plot(epochs_ran, val_mae, linewidth=2, label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("Scaled MAE")
plt.title("LSTM Training vs Validation MAE")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 13. GENERALIZATION GAP
# ============================================================
best_index = int(np.argmin(val_loss))
train_loss_best = loss[best_index]
val_loss_best = val_loss[best_index]

gap = val_loss_best - train_loss_best
gap_ratio = gap / max(abs(val_loss_best), 1e-12)

print("Overfitting diagnostics")
print("-----------------------")
print(f"Best epoch: {best_epoch}")
print(f"Training loss at best epoch: {train_loss_best:.6f}")
print(f"Validation loss at best epoch: {val_loss_best:.6f}")
print(f"Absolute generalization gap: {gap:.6f}")
print(f"Relative gap: {gap_ratio:.2%}")

if gap_ratio < 0.10:
    print("Interpretation: small train/validation gap.")
elif gap_ratio < 0.25:
    print("Interpretation: moderate train/validation gap.")
else:
    print("Interpretation: large train/validation gap; inspect the curves carefully.")


In [ ]:
# ============================================================
# 14. LOAD BEST MODEL
# ============================================================
best_model = load_model(MODEL_PATH)
print("Best model loaded.")


In [ ]:
# ============================================================
# 15. PREDICT ATLANTA INTERNAL TEST + BIRMINGHAM EXTERNAL TEST
# ============================================================
atlanta_pred_scaled = best_model.predict(
    X_atlanta_test, batch_size=BATCH_SIZE, verbose=1
)

birmingham_pred_scaled = best_model.predict(
    X_birmingham_test, batch_size=BATCH_SIZE, verbose=1
)

print("Predictions generated.")


In [ ]:
# ============================================================
# 16. CONVERT BACK TO MW USING ATLANTA y SCALER
# ============================================================
y_atlanta_actual = y_scaler.inverse_transform(
    y_atlanta_test.reshape(-1, 1)
).ravel()

y_atlanta_pred = y_scaler.inverse_transform(
    atlanta_pred_scaled
).ravel()

y_birmingham_actual = y_scaler.inverse_transform(
    y_birmingham_test.reshape(-1, 1)
).ravel()

y_birmingham_pred = y_scaler.inverse_transform(
    birmingham_pred_scaled
).ravel()

print("Predictions converted back to MW.")


In [ ]:
# ============================================================
# 17. METRICS
# ============================================================
def calculate_metrics(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)

    nonzero = np.abs(actual) > 1e-8
    mape = (
        np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero])) * 100
        if nonzero.any() else np.nan
    )

    return {
        "MAE_MW": mae,
        "RMSE_MW": rmse,
        "R2": r2,
        "MAPE_percent": mape
    }

atlanta_metrics = calculate_metrics(y_atlanta_actual, y_atlanta_pred)
birmingham_metrics = calculate_metrics(y_birmingham_actual, y_birmingham_pred)

metrics_df = pd.DataFrame(
    [atlanta_metrics, birmingham_metrics],
    index=["Atlanta Internal Test", "Birmingham External Test"]
)

metrics_df


In [ ]:
# ============================================================
# 18. ATLANTA ACTUAL VS PREDICTED LINE
# ============================================================
n_plot = min(500, len(y_atlanta_actual))

plt.figure(figsize=(13, 5))
plt.plot(y_atlanta_actual[:n_plot], linewidth=2, label="Actual load")
plt.plot(y_atlanta_pred[:n_plot], linewidth=2, label="Predicted load")
plt.xlabel("Test observation")
plt.ylabel("Load (MW)")
plt.title("Atlanta: Actual vs Predicted Load")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 19. ATLANTA ACTUAL VS PREDICTED SCATTER
# ============================================================
plt.figure(figsize=(7, 7))
plt.scatter(y_atlanta_actual, y_atlanta_pred, alpha=0.25, s=18)

lo = min(y_atlanta_actual.min(), y_atlanta_pred.min())
hi = max(y_atlanta_actual.max(), y_atlanta_pred.max())

plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=2, label="Perfect prediction")
plt.xlabel("Actual load (MW)")
plt.ylabel("Predicted load (MW)")
plt.title("Atlanta: Actual vs Predicted")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 20. BIRMINGHAM EXTERNAL TEST ACTUAL VS PREDICTED LINE
# ============================================================
n_plot = min(500, len(y_birmingham_actual))

plt.figure(figsize=(13, 5))
plt.plot(y_birmingham_actual[:n_plot], linewidth=2, label="Actual load")
plt.plot(y_birmingham_pred[:n_plot], linewidth=2, label="Predicted load")
plt.xlabel("External test observation")
plt.ylabel("Load (MW)")
plt.title("Birmingham External Test: Atlanta-Trained LSTM")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 21. BIRMINGHAM EXTERNAL TEST SCATTER
# ============================================================
plt.figure(figsize=(7, 7))
plt.scatter(y_birmingham_actual, y_birmingham_pred, alpha=0.25, s=18)

lo = min(y_birmingham_actual.min(), y_birmingham_pred.min())
hi = max(y_birmingham_actual.max(), y_birmingham_pred.max())

plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=2, label="Perfect prediction")
plt.xlabel("Actual load (MW)")
plt.ylabel("Predicted load (MW)")
plt.title("Birmingham External Test: Actual vs Predicted")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 22. ATLANTA RESIDUALS
# ============================================================
atlanta_residuals = y_atlanta_actual - y_atlanta_pred

plt.figure(figsize=(12, 5))
plt.scatter(np.arange(len(atlanta_residuals)), atlanta_residuals, alpha=0.25, s=18)
plt.axhline(0, linestyle="--", linewidth=2)
plt.xlabel("Test observation")
plt.ylabel("Residual (MW)")
plt.title("Atlanta Residual Analysis")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 23. BIRMINGHAM RESIDUALS
# ============================================================
birmingham_residuals = y_birmingham_actual - y_birmingham_pred

plt.figure(figsize=(12, 5))
plt.scatter(np.arange(len(birmingham_residuals)), birmingham_residuals, alpha=0.25, s=18)
plt.axhline(0, linestyle="--", linewidth=2)
plt.xlabel("External test observation")
plt.ylabel("Residual (MW)")
plt.title("Birmingham External Test Residual Analysis")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 24. RESIDUAL DISTRIBUTIONS
# ============================================================
plt.figure(figsize=(11, 5))
plt.hist(atlanta_residuals, bins=60, alpha=0.45, density=True, label="Atlanta residuals")
plt.hist(birmingham_residuals, bins=60, alpha=0.45, density=True, label="Birmingham residuals")
plt.axvline(0, linestyle="--", linewidth=2)
plt.xlabel("Residual (MW)")
plt.ylabel("Density")
plt.title("Residual Distribution: Internal vs External Test")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 25. SAVE PREDICTIONS
# ============================================================
pd.DataFrame({
    "actual_load_mw": y_atlanta_actual,
    "predicted_load_mw": y_atlanta_pred,
    "residual_mw": atlanta_residuals
}).to_csv(ATLANTA_PREDICTIONS_PATH, index=False)

pd.DataFrame({
    "actual_load_mw": y_birmingham_actual,
    "predicted_load_mw": y_birmingham_pred,
    "residual_mw": birmingham_residuals
}).to_csv(BIRMINGHAM_PREDICTIONS_PATH, index=False)

print("Saved Atlanta predictions:", ATLANTA_PREDICTIONS_PATH)
print("Saved Birmingham predictions:", BIRMINGHAM_PREDICTIONS_PATH)


# Final interpretation

**Atlanta internal test:** evaluates generalization to unseen data from the same city.

**Birmingham external test:** evaluates cross-city generalization. Birmingham was not used during model fitting or model selection.

A difference between Atlanta and Birmingham performance can represent either model overfitting or genuine distribution shift. Use the training/validation curves together with the two test results to distinguish these cases.

For the professor presentation, the most useful visuals are:
1. Training vs validation loss
2. Atlanta actual vs predicted line
3. Atlanta actual vs predicted scatter
4. Birmingham external actual vs predicted line
5. Birmingham external scatter
6. Residual plots

No bar/block charts are used in this notebook.
